# Dataset benchmark view

This notebook is for benchmark runs that include named datasets. It avoids the D/N/K heatmap assumptions used by the synthetic sweep notebook and groups results primarily by `Dataset`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PYTHON_SRC = Path.cwd() / "python"
if str(PYTHON_SRC) not in sys.path:
    sys.path.insert(0, str(PYTHON_SRC))

from benchmark_reporting.constants import *
from benchmark_reporting.io import (
    load_benchmark_summary,
    load_benchmark_data,
    load_cachegrind_summary,
    load_exclusion_summary,
    load_speedup_summary,
)

SUMMARY_JSON = Path("datasets/benchmark_summary.json")


In [ ]:
summary = load_benchmark_summary(SUMMARY_JSON)
configs = pd.DataFrame(summary.get("configs", []))

if configs.empty:
    display(Markdown("No configs found in the benchmark summary."))
else:
    overview = (
        configs[["dataset", "dimensions", "samples", "clusters", "config_id"]]
        .rename(
            columns={
                "dataset": COL_DATASET,
                "dimensions": COL_DIMENSIONS,
                "samples": COL_SAMPLES,
                "clusters": COL_CLUSTERS,
                "config_id": "Config ID",
            }
        )
        .drop_duplicates()
        .sort_values([COL_DATASET, COL_DIMENSIONS, COL_SAMPLES, COL_CLUSTERS])
        .reset_index(drop=True)
    )
    display(Markdown("## Dataset/config overview"))
    display(overview)


In [ ]:
benchmarks = load_benchmark_data(SUMMARY_JSON)

if benchmarks.empty:
    display(Markdown("No benchmark timing rows found."))
else:
    timing_cols = [
        COL_DATASET,
        COL_PHASE,
        COL_STAGE,
        COL_VARIANT,
        COL_PARAMS,
        COL_REFERENCE,
        COL_LANGUAGE,
        COL_DIMENSIONS,
        COL_SAMPLES,
        COL_CLUSTERS,
        COL_TIME_S,
    ]
    display(Markdown("## Timing rows by dataset"))
    display(
        benchmarks[timing_cols]
        .sort_values([COL_DATASET, COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS, COL_LANGUAGE])
        .reset_index(drop=True)
    )


In [ ]:
speedups = load_speedup_summary(SUMMARY_JSON)

# Backward compatibility for older summaries.
if not speedups.empty and COL_DATASET not in speedups.columns:
    speedups[COL_DATASET] = "blobs"

required_cols = {
    COL_DATASET,
    COL_PHASE,
    COL_SPEEDUP,
    COL_SPEEDUP_CI_LOW,
    COL_SPEEDUP_CI_HIGH,
}

if not speedups.empty and required_cols.issubset(speedups.columns):
    plot_data = (
        speedups.groupby([COL_DATASET, COL_PHASE], observed=True)
        .agg(
            speedup=(COL_SPEEDUP, "median"),
            ci_low=(COL_SPEEDUP_CI_LOW, "median"),
            ci_high=(COL_SPEEDUP_CI_HIGH, "median"),
        )
        .reset_index()
        .sort_values([COL_DATASET, COL_PHASE])
    )

    if not plot_data.empty:
        values = plot_data.pivot(
            index=COL_DATASET,
            columns=COL_PHASE,
            values="speedup",
        )

        ci_low = plot_data.pivot(
            index=COL_DATASET,
            columns=COL_PHASE,
            values="ci_low",
        ).reindex_like(values)

        ci_high = plot_data.pivot(
            index=COL_DATASET,
            columns=COL_PHASE,
            values="ci_high",
        ).reindex_like(values)

        x = np.arange(len(values.index))
        phases = list(values.columns)
        width = 0.8 / max(len(phases), 1)

        fig, ax = plt.subplots(figsize=(10, 5))

        for i, phase in enumerate(phases):
            y = values[phase].to_numpy(dtype=float)
            low = ci_low[phase].to_numpy(dtype=float)
            high = ci_high[phase].to_numpy(dtype=float)

            lower_err = np.maximum(y - low, 0.0)
            upper_err = np.maximum(high - y, 0.0)

            offset = (i - (len(phases) - 1) / 2) * width

            ax.bar(
                x + offset,
                y,
                width,
                label=phase,
                yerr=np.vstack([lower_err, upper_err]),
                capsize=4,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(values.index, rotation=45, ha="right")
        ax.set_ylabel(COL_SPEEDUP)
        ax.set_title("Median speedup by dataset and phase")
        ax.axhline(1.0, linewidth=1)
        ax.legend(title=COL_PHASE)
        fig.tight_layout()
else:
    missing = sorted(required_cols - set(speedups.columns))
    print(f"Cannot plot speedup CI bars. Missing columns: {missing}")


In [ ]:
cachegrind = load_cachegrind_summary(SUMMARY_JSON)

if cachegrind.empty:
    display(Markdown("No Cachegrind rows found."))
else:
    cachegrind_cols = [
        COL_DATASET,
        COL_PHASE,
        COL_STAGE,
        COL_VARIANT,
        COL_PARAMS,
        COL_DIMENSIONS,
        COL_SAMPLES,
        COL_CLUSTERS,
        COL_CACHEGRIND_IR,
        COL_CACHEGRIND_D1_DATA_MISS_RATE,
        COL_CACHEGRIND_LL_DATA_MISS_RATE,
    ]
    available_cols = [col for col in cachegrind_cols if col in cachegrind.columns]
    display(Markdown("## Cachegrind by dataset"))
    display(
        cachegrind[available_cols]
        .sort_values([COL_DATASET, COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS])
        .reset_index(drop=True)
    )


In [ ]:
exclusions = load_exclusion_summary(SUMMARY_JSON)

if exclusions.empty:
    display(Markdown("No configured exclusions found."))
else:
    display(Markdown("## Exclusions by dataset"))
    display(
        exclusions.sort_values([COL_DATASET, COL_PHASE, COL_STAGE, COL_DIMENSIONS, COL_SAMPLES, COL_CLUSTERS])
        .reset_index(drop=True)
    )
